In [1]:
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
REPO_ROOT = "/home/jorge.gacitua/datosmunin/wrf_python_Assimilation/fortran"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from cletkf_wloc import common_da as cda

def plot_error_analysis(data1, data2, data3,figname):
    # Define file paths
    file_paths = {
        'LETKF'   : data1,
        'LETKF-T2': data2,
        'LETKF-T3': data3
    }

    # Variable names
    variables = {
        0: 'Q Graupel',
        3: 'Temperature',
        7: 'Vertical Wind'
    }

    fig, axs = plt.subplots(3, 3, figsize=(18, 12))
    for i, (var_index, var_name) in enumerate(variables.items()):
        contour_levels = {}  # Store contour levels and ticks per row
        err_maxs = []
        for col_index, (col_name, data) in enumerate(file_paths.items()):
            xa = data['xa'][:, 0, :, :, :]
            true_state = data['truth'][:, 0, :, :]
            mean_xa = np.nanmean(xa, axis=2)
            analysis_field = mean_xa[:, :, var_index]
            true_field = true_state[:, :, var_index]
            idx_zero = true_field == 0
            true_field[idx_zero] = np.nan  # Set to NaN to avoid division by zero
            error_analysis = analysis_field - true_field        #print(f'Error analysis shape: {error_analysis}')
            err_maxs.append(np.nanmax(np.abs(error_analysis)))

        err_max = np.nanmax(err_maxs)
        contour_levels['error'] = np.linspace(-err_max, err_max, 20)

        for col_index, (col_name, data) in enumerate(file_paths.items()):
            xa = data['xa'][:, 0, :, :, :]
            true_state = data['truth'][:, 0, :, :]
            mean_xa = np.nanmean(xa, axis=2)
            analysis_field = mean_xa[:, :, var_index]
            true_field = true_state[:, :, var_index]
            # Avoid division by zero

            error_analysis = analysis_field - true_field
            # Analysis Error
            im = axs[i, col_index].contourf(error_analysis.T, cmap='RdBu_r', levels=contour_levels['error'])
            cb2 = fig.colorbar(im, ax=axs[i, col_index])
            rmse = np.sqrt(mean_squared_error(true_field, analysis_field))
            axs[i, col_index].set_title(f"{col_name} | RMSE: {rmse:.3f} ")

            # Grid every 2 units and add square
            for ax in axs[:, col_index]:
                ax.grid(True, which='major', linestyle='--', linewidth=0.6, alpha=0.6)
                #ax.set_xticks(np.arange(0, forecast_field.shape[0], 5))
                #ax.set_yticks(np.arange(0, forecast_field.shape[1], 5))
            if col_index == 0:
                axs[i, col_index].set_ylabel(f"{var_name}", fontsize=14)
        # General title
        #fig.suptitle(f"Variable: {var_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(figname, dpi=300)
    plt.close()
     
def tight_bbox_from_field(field2d, threshold=0.0, pad=2):
    """
    Return (xmin, xmax, zmin, zmax) axis limits in index space for
    the region where field2d > threshold. pad is in grid cells.
    """
    a = np.asarray(field2d)
    mask = a > threshold
    if not np.any(mask):
        # fall back to full domain
        xn, zn = a.shape
        return -0.5, xn - 0.5, -0.5, zn - 0.5
    ix, iz = np.where(mask)           # ix ~ x-index (first dim), iz ~ z-index (second dim)
    xn, zn = a.shape
    xmin = max(ix.min() - pad, 0)
    xmax = min(ix.max() + pad, xn - 1)
    zmin = max(iz.min() - pad, 0)
    zmax = min(iz.max() + pad, zn - 1)
    # use half-cell edges for pretty bounds
    return xmin - 0.5, xmax + 0.5, zmin - 0.5, zmax + 0.5

def plot_observations(data,figname):
    truth = data['truth'][:]
    xa    = data['xa'][:]
    xn,yn,zn = xa.shape[0:3]
    yo1    = data['yo'][:]
    atemp1 = data['steps'][:]
    ox1,oy1,oz1 = data['ox'][:], data['oy'][:], data['oz'][:]
    deps1 = data['deps'][:]
    ref = np.zeros((xn,yn,zn))
    for xi in np.arange(xn):
        for yi in np.arange(yn):
            for zi in np.arange(zn):
                qr = truth[xi, yi, zi, 1]
                qs = truth[xi, yi, zi, 2]
                qg = truth[xi, yi, zi, 0]
                tt = truth[xi, yi, zi, 3]
                pp = truth[xi, yi, zi, 4]

                ref[xi,yi,zi] = cda.calc_ref(qr, qs, qg, tt, pp)
    ref2d = ref[:,0,:]
    xmin, xmax, zmin, zmax = tight_bbox_from_field(ref2d, threshold=5.0, pad=2)

    fig, ax = plt.subplots(1, 3, figsize=(19, 3),dpi=300)
    im=ax[0].scatter(ox1,oz1,c=yo1,cmap='Spectral_r', marker='o', s=35,vmin=5,vmax=40)

    fig.colorbar(im, ax=ax[0], label="Reflectivity (dBZ)")
    ax[0].set_title(f'Observations  $[Nobs = {len(yo1)}]$')
    ax[0].set_aspect('equal','box')

    X, Z = np.meshgrid(np.arange(xn), np.arange(zn), indexing='ij')
    #im3 = ax[1].scatter(X.flatten(), Z.flatten(), c=ref[:,0,:].flatten(), cmap='Spectral_r', marker='o', s=10,vmin=0,vmax=50)
    im2 = ax[1].pcolormesh(X, Z, ref2d, cmap="Spectral_r", vmin=5, vmax=40, shading='auto')
    cd  = ax[1].scatter(ox1, oz1, s=12, color="white", alpha=0.6, edgecolor="none", rasterized=True)
    fig.colorbar(im2, ax=ax[1], label="Reflectivity (dBZ)")
    ax[1].set_title('Truth reflectivity $[Hxf]$')
    ax[1].set_aspect('equal','box')

    im=ax[2].scatter(ox1,oz1,c=deps1,cmap='RdBu_r', marker='o', edgecolor="k", s=35,vmin=-20,vmax=20)

    fig.colorbar(im, ax=ax[2], label="Reflectivity (dBZ)")
    ax[2].set_title('Departure $[y0 - \overline{ H[x_f] }]$')
    ax[2].set_aspect('equal','box')

    xmin, xmax, zmin, zmax = tight_bbox_from_field(ref2d, threshold=5.0, pad=2)

    for a in ax:
        a.set_xlim(xmin, xmax)
        a.set_ylim(zmin, zmax)
        a.set_aspect('equal', 'box')
        cs = a.contour(ref2d.T,levels = [5,10], colors="k", linewidths=0.8)
        a.clabel(cs, fmt="%.0f", inline=True, fontsize=8)
    plt.tight_layout()
    plt.savefig(figname, dpi=300)
    plt.close()



In [2]:


path = "/home/jorge.gacitua/datosmunin/wrf_python_Assimilation/data/runs/full2d_multicycle/"
figpath_error = "/home/jorge.gacitua/datosmunin/wrf_python_Assimilation/figures/error/"
figpath_obs = "/home/jorge.gacitua/datosmunin/wrf_python_Assimilation/figures/observations/"
alpha_range = [0,1,2,3]
loc_range = [-1,1,2,3,4,5,6,7,8,9,10]
nens_range = np.arange(30)

for alpha in alpha_range:
    for loc in loc_range:
        for true in nens_range:
            try:
                
                path1 =  f'{path}Multicycle_v1_2023-12-16_19:00:00_temp1_alpha{alpha}_Loc{loc}_True{true}_kindFULL_2D.npz'
                path2 =  f'{path}Multicycle_v1_2023-12-16_19:00:00_temp2_alpha{alpha}_Loc{loc}_True{true}_kindFULL_2D.npz'
                path3 =  f'{path}Multicycle_v1_2023-12-16_19:00:00_temp3_alpha{alpha}_Loc{loc}_True{true}_kindFULL_2D.npz'

                data1 = np.load(path1)
                data2 = np.load(path2)
                data3 = np.load(path3)
                figname_error = f'Error_Analysis_alpha{alpha}_Loc{loc}_True{true}.png'
                plot_error_analysis(data1, data2, data3,figpath_error+figname_error)
                figname_obs = f'Observations_alpha{alpha}_Loc{loc}_True{true}.png'
                plot_observations(data1,figpath_obs+figname_obs)
                print(f'alpha {alpha} loc {loc} true {true}')
            except Exception as e:
                #print(f"Error processing alpha {alpha}, loc {loc}, true {true}: {e}")
                continue

alpha 0 loc -1 true 0
alpha 0 loc -1 true 1
alpha 0 loc -1 true 2
alpha 0 loc -1 true 3
alpha 0 loc -1 true 4
alpha 0 loc -1 true 5
alpha 0 loc -1 true 6
alpha 0 loc -1 true 7
alpha 0 loc -1 true 8
alpha 0 loc -1 true 9
alpha 0 loc -1 true 10
alpha 0 loc -1 true 11
alpha 0 loc -1 true 12
alpha 0 loc -1 true 13
alpha 0 loc -1 true 14
alpha 0 loc -1 true 15
alpha 0 loc -1 true 16
alpha 0 loc -1 true 17
alpha 0 loc -1 true 18
alpha 0 loc -1 true 19
alpha 0 loc -1 true 20
alpha 0 loc -1 true 21
alpha 0 loc -1 true 22
alpha 0 loc -1 true 23
alpha 0 loc -1 true 24
alpha 0 loc -1 true 25
alpha 0 loc -1 true 26
alpha 0 loc -1 true 27
alpha 0 loc -1 true 28
alpha 0 loc -1 true 29
alpha 0 loc 1 true 0
alpha 0 loc 1 true 1
alpha 0 loc 1 true 2
alpha 0 loc 1 true 3
alpha 0 loc 1 true 4
alpha 0 loc 1 true 5
alpha 0 loc 1 true 6
alpha 0 loc 1 true 7
alpha 0 loc 1 true 8
alpha 0 loc 1 true 9
alpha 0 loc 1 true 10
alpha 0 loc 1 true 11
alpha 0 loc 1 true 12
alpha 0 loc 1 true 13
alpha 0 loc 1 true 14
